# 06· Modelo de Machine Learning: Random Forest con rezagos 

Este notebook entrena el modelo de machine learning del proyecto: un **Random Forest** que pronostica la varianza diaria del WTI (estimador de Parkinson del día τ) a partir de rezagos y medidas móviles de volatilidad, con información disponible hasta τ−1. Se evalúa en los mismos 10 folds de Time Series Cross-Validation que el modelo econométrico AR-EGARCH, para compararlos en exactamente los mismos días.

**Contenido**

0. Configuración
1. Carga de datos, folds y selección de variables
2. Especificación del modelo y del pipeline
3. Funciones de evaluación
4. Optimización de hiperparámetros con Optuna
5. Gráficas de la optimización: historial, curva de validación y curva de aprendizaje
6. Errores del modelo final por fold (train y validación)
7. Gráficas de resultados
8. Guardar resultados
9. Conclusiones

## 0. Configuración

In [2]:
import sys
import warnings
from pathlib import Path

In [3]:
# Raíz del proyecto (funciona desde la raíz o desde notebooks/)
RAIZ = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ))
from src import config as cfg

In [9]:


import importlib
import re
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from arch import arch_model
from src import ar_garch as ag

from arch.univariate import ARX, EGARCH, StudentsT


In [10]:
importlib.reload(ag)

<module 'src.ar_garch' from 'c:\\Users\\ASUS\\Desktop\\Maestria IA USFQ\\AprendizajeSupervisado\\Proyectofinal_ml_wti\\src\\ar_garch.py'>

In [13]:
PROC = RAIZ / "data" / "processed"
RES_DIR = RAIZ / "results"
FIG_DIR = RAIZ / "figures"
RES_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
FS_DIR = PROC / "feature_selection"


### Configuración de parámetros de visualización
plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True, "grid.alpha": 0.3})
warnings.filterwarnings("ignore", category=FutureWarning)

## 1. Carga de datos, folds y selección de variables


- `X`: 67 variables candidatas construidas por extracción de rezagos (notebook 04). Cada fila τ usa información hasta τ−1.
- `y`: estimador de Parkinson del día τ (%²), variable objetivo.
- `FOLDS`: 10 folds de Time Series Cross-Validation (2 repeticiones × 5 folds, ventana expansiva).
- `K_FINAL`: número de variables elegido en la selección de variables (notebook 05).
- `MI_FOLD`: información mutua de cada variable calculada con el entrenamiento de cada fold.

In [15]:

X = pd.read_csv(PROC / "fe_features_X.csv", index_col=0, parse_dates=True)

y_df = pd.read_csv(PROC / "fe_features_y.csv", index_col=0, parse_dates=True)
y = y_df.iloc[:, 0]
y.name = "WTI_park"

X, y = X.align(y, join="inner", axis=0)     # mismas fechas y mismo orden

# 2. Folds de Time Series Cross-Validation
FOLDS = pd.read_csv(PROC / "folds.csv",
                    parse_dates=["train_ini", "train_fin", "val_ini", "val_fin"])

In [16]:

config_fs = json.loads((FS_DIR / "fs_config.json").read_text(encoding="utf-8"))
K_FINAL = config_fs["k"]                                            # 16

mi_por_fold = pd.read_csv(FS_DIR / "fs_mi_por_fold.csv", index_col=0)
MI_FOLD = [mi_por_fold[c].reindex(X.columns).to_numpy() for c in mi_por_fold.columns]

In [17]:
assert set(mi_por_fold.index) == set(X.columns), "X no tiene las mismas columnas que en feature selection"
assert len(MI_FOLD) == len(FOLDS)
assert X.notna().all().all() and y.notna().all()

print(f"X: {X.shape} | y: {y.shape} | {X.index.min().date()} → {X.index.max().date()}")
print(f"k (feature selection): {K_FINAL} | folds: {len(FOLDS)}")

X: (3706, 67) | y: (3706,) | 2010-04-07 → 2024-12-31
k (feature selection): 16 | folds: 10


## 2. Especificación del modelo y del pipeline

**Random Forest.** Conjunto de árboles de regresión entrenados sobre muestras *bootstrap* del entrenamiento; cada árbol considera en cada división una fracción aleatoria de las variables. El pronóstico es el promedio de los árboles, lo que reduce la varianza frente a un único árbol y permite capturar relaciones no lineales entre la volatilidad pasada y la futura.

**Pipeline.** El modelo se implementa como un `Pipeline` de scikit-learn con dos pasos:

1. `SelectKBest(mutual_info_regression, k=16)`: selección tipo *filter* de las 16 variables con mayor información mutua.
2. `RandomForestRegressor`: modelo de regresión.

En cada fold, `pipeline.fit(X_train, y_train)` ajusta **ambos pasos solo con el entrenamiento**: la información mutua se calcula con esos días y el Random Forest se entrena con las variables elegidas. Luego `pipeline.predict(X_val)` aplica la misma selección y el modelo ya entrenado a la validación, sin volver a ajustar nada. Así no hay fuga de información desde la validación.

**Variable objetivo.** Se entrena directamente sobre el Parkinson del día τ (%²), sin transformación logarítmica. El MSE y la QLIKE se minimizan cuando el pronóstico es la media condicional de la varianza; con logaritmo, el modelo estimaría la media de ln(y) y al deshacer la transformación subestimaría la varianza de forma sistemática. La influencia de los días extremos (2020) se controla con `min_samples_leaf`.

**Funciones de pérdida** (en cada día objetivo τ de la validación):
- $\text{MSE} = (\sigma^2_{P,\tau} - \hat\sigma^2_\tau)^2$
- $\text{QLIKE} = \dfrac{\sigma^2_{P,\tau}}{\hat\sigma^2_\tau} + \ln \hat\sigma^2_\tau$, robusta al ruido del proxy (Patton, 2011). Es la métrica principal.

## 3. Funciones de evaluación

- `perdidas`: MSE y QLIKE diarios.
- `crear_pipeline`: arma el pipeline selección + Random Forest con los hiperparámetros indicados.
- `CV`: índices de entrenamiento y validación de cada fold, definidos por fechas (los mismos días que el EGARCH).
- `puntajes_fijos`: permite que `SelectKBest` use la información mutua ya calculada en el notebook 05 durante la búsqueda de Optuna. Es idéntico a recalcularla, pero evita repetir el mismo cálculo en cada prueba.

In [18]:
def perdidas(y_true, h):
    # MSE y QLIKE diarios (QLIKE robusta de Patton, 2011)
    h = np.clip(np.asarray(h, dtype=float), H_MIN, None)
    y_true = np.asarray(y_true, dtype=float)
    return pd.DataFrame({"mse": (y_true - h)**2, "qlike": y_true / h + np.log(h)})


def crear_pipeline(k, score_func=None, **params_rf):
    # Pipeline: selección tipo filter (información mutua) + Random Forest
    return Pipeline([
        ("seleccion", SelectKBest(score_func=score_func or mi_score, k=k)),
        ("modelo", RandomForestRegressor(random_state=SEED, n_jobs=-1, **params_rf)),
    ])


def puntajes_fijos(scores):
    # score_func para SelectKBest que devuelve puntajes ya calculados
    return lambda X_, y_: np.asarray(scores)


def mascara(index, ini, fin):
    return (index >= ini) & (index <= fin)


# Índices de entrenamiento y validación de cada fold (por fechas)
CV = []
for f in FOLDS.itertuples():
    tr = np.where(mascara(X.index, f.train_ini, f.train_fin))[0]
    va = np.where(mascara(X.index, f.val_ini, f.val_fin))[0]
    CV.append((f.rep, f.fold, tr, va))

for rep, fold, tr, va in CV:
    print(f"R{rep}·F{fold}: train {X.index[tr[0]].date()} → {X.index[tr[-1]].date()} ({len(tr)}) | "
          f"val {X.index[va[0]].date()} → {X.index[va[-1]].date()} ({len(va)})")

R1·F1: train 2010-04-07 → 2019-12-31 (2450) | val 2020-01-02 → 2020-12-31 (251)
R1·F2: train 2010-04-07 → 2020-12-31 (2701) | val 2021-01-04 → 2021-12-31 (252)
R1·F3: train 2010-04-07 → 2021-12-31 (2953) | val 2022-01-03 → 2022-12-30 (251)
R1·F4: train 2010-04-07 → 2022-12-30 (3204) | val 2023-01-03 → 2023-12-29 (250)
R1·F5: train 2010-04-07 → 2023-12-29 (3454) | val 2024-01-02 → 2024-12-31 (252)
R2·F1: train 2010-04-07 → 2019-06-28 (2322) | val 2019-07-01 → 2020-06-30 (251)
R2·F2: train 2010-04-07 → 2020-06-30 (2573) | val 2020-07-01 → 2021-06-30 (252)
R2·F3: train 2010-04-07 → 2021-06-30 (2825) | val 2021-07-01 → 2022-06-30 (252)
R2·F4: train 2010-04-07 → 2022-06-30 (3077) | val 2022-07-01 → 2023-06-30 (251)
R2·F5: train 2010-04-07 → 2023-06-30 (3328) | val 2023-07-03 → 2024-06-28 (250)


##  4. Optimización de hiperparámetros con Optuna

Se buscan los hiperparámetros del Random Forest que minimizan la **QLIKE promedio de validación en los 10 folds**. Esta validación cruzada para series de tiempo (ventana expansiva, 2 repeticiones × 5 folds) sustituye al *repeated k-fold* tradicional, que no es válido en series de tiempo porque usaría observaciones futuras para entrenar. El número de variables se mantiene en k = 16, elegido con la misma validación en el notebook de selección de variables.

| Hiperparámetro | Rango | Qué controla |
|---|---|---|
| `n_estimators` | 100 – 600 | Número de árboles |
| `max_depth` | 2 – 20 | Profundidad máxima de cada árbol (complejidad del modelo) |
| `min_samples_leaf` | 5 – 200 (escala log) | Mínimo de días por hoja; suaviza el efecto de los días extremos |
| `max_features` | 0.2 – 1.0 | Fracción de variables evaluadas en cada división (diversidad entre árboles) |

En cada prueba, y para cada fold, el pipeline se ajusta con `.fit` sobre el entrenamiento y se evalúa con `.predict` sobre la validación. Se usa el muestreador TPE (búsqueda bayesiana) y poda por mediana: si una prueba va peor que la mediana de las anteriores después de varios folds, se detiene para ahorrar tiempo. El estudio se guarda en una base SQLite, de modo que la búsqueda puede retomarse si se interrumpe.

In [19]:
def objetivo(trial):
    params = dict(
        n_estimators=trial.suggest_int("n_estimators", 100, 600, step=50),
        max_depth=trial.suggest_int("max_depth", 2, 20),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 5, 200, log=True),
        max_features=trial.suggest_float("max_features", 0.2, 1.0),
    )
    errores = []
    for i, (rep, fold, tr, va) in enumerate(CV):
        pipe = crear_pipeline(K_FINAL, score_func=puntajes_fijos(MI_FOLD[i]), **params)
        pipe.fit(X.iloc[tr], y.iloc[tr])                         # ajuste solo con el entrenamiento
        h_va = pipe.predict(X.iloc[va])                          # evaluación en la validación
        errores.append(perdidas(y.iloc[va], h_va)["qlike"].mean())

        trial.report(np.mean(errores), step=i)                   # poda si va peor que la mediana
        if trial.should_prune():
            raise optuna.TrialPruned()
    return float(np.mean(errores))

In [20]:
N_TRIALS = 50

study = optuna.create_study(
    direction="minimize",
    study_name="rf_volatilidad_wti",
    storage=f"sqlite:///{RES_DIR / 'optuna_rf.db'}",
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=3),
)
pendientes = max(0, N_TRIALS - len(study.trials))
t0 = time.time()
study.optimize(objetivo, n_trials=pendientes, show_progress_bar=True)
print(f"Tiempo: {(time.time() - t0) / 60:.1f} min | pruebas: {len(study.trials)}")

estados = pd.Series([t.state.name for t in study.trials]).value_counts()
display(estados.to_frame("pruebas"))
print(f"Mejor QLIKE promedio en validación: {study.best_value:.4f}")

best = dict(study.best_params)
display(pd.Series({"k": K_FINAL, **best}, name="mejores hiperparámetros").to_frame())

NameError: name 'optuna' is not defined

## 5. Gráficas de la optimización


- **Historial:** QLIKE de cada prueba y mejor valor acumulado; muestra que la búsqueda converge.
- **Importancia de los hiperparámetros:** cuáles influyen más en el error de validación.
- **Slice:** QLIKE frente a cada hiperparámetro, para ver en qué zona se ubican los mejores valores.

### Gráficas de Optuna

In [ ]:
from optuna.visualization.matplotlib import (plot_optimization_history, plot_param_importances,
                                             plot_slice)

ax = plot_optimization_history(study)
ax.set_title("Historial de la optimización (QLIKE promedio en los 10 folds)")
plt.gcf().set_size_inches(10, 4); plt.tight_layout()
plt.savefig(FIG_DIR / "06_optuna_historial.png", dpi=150); plt.show()

ax = plot_param_importances(study)
plt.gcf().set_size_inches(8, 4); plt.tight_layout()
plt.savefig(FIG_DIR / "06_optuna_importancia_hiperparametros.png", dpi=150); plt.show()

axes = plot_slice(study, params=["n_estimators", "max_depth", "min_samples_leaf", "max_features"])
plt.gcf().set_size_inches(16, 4); plt.tight_layout()
plt.savefig(FIG_DIR / "06_optuna_slice.png", dpi=150); plt.show()

### Curva de validación: profundidad de los árboles


Con los demás hiperparámetros fijos en los mejores valores, se varía `max_depth` y se calcula la QLIKE en entrenamiento y validación (folds de la repetición 1). Árboles poco profundos con error alto en ambos indican **subajuste**; si al aumentar la profundidad el error de entrenamiento sigue bajando pero el de validación no mejora, hay **sobreajuste**.

In [21]:
PROF = [2, 3, 4, 6, 8, 10, 12, 15, 20]
folds_curva = [(i, c) for i, c in enumerate(CV) if c[0] == 1]     # repetición 1

filas = []
for d in PROF:
    params_d = dict(best, max_depth=d)
    for i, (rep, fold, tr, va) in folds_curva:
        pipe = crear_pipeline(K_FINAL, score_func=puntajes_fijos(MI_FOLD[i]), **params_d)
        pipe.fit(X.iloc[tr], y.iloc[tr])
        filas.append({"max_depth": d, "fold": fold,
                      "train": perdidas(y.iloc[tr], pipe.predict(X.iloc[tr]))["qlike"].mean(),
                      "validación": perdidas(y.iloc[va], pipe.predict(X.iloc[va]))["qlike"].mean()})

curva_prof = pd.DataFrame(filas).groupby("max_depth")[["train", "validación"]].agg(["mean", "std"])
display(curva_prof.round(4))

fig, ax = plt.subplots(figsize=(9, 4.5))
for parte, color in [("train", "tab:blue"), ("validación", "tab:orange")]:
    m, s = curva_prof[(parte, "mean")], curva_prof[(parte, "std")]
    ax.plot(m.index, m, "o-", color=color, label=parte)
    ax.fill_between(m.index, m - s, m + s, color=color, alpha=0.15)
ax.axvline(best["max_depth"], color="grey", ls="--", label=f"elegido por Optuna ({best['max_depth']})")
ax.set_xlabel("max_depth"); ax.set_ylabel("QLIKE promedio")
ax.set_title("Curva de validación: profundidad máxima de los árboles")
ax.legend(); plt.tight_layout(); plt.savefig(FIG_DIR / "06_curva_validacion_max_depth.png", dpi=150); plt.show()

NameError: name 'best' is not defined

### Curvas de aprendizaje

In [22]:
FRACCIONES = [0.2, 0.35, 0.5, 0.65, 0.8, 1.0]

filas = []
for frac in FRACCIONES:
    for i, (rep, fold, tr, va) in folds_curva:
        n = max(int(len(tr) * frac), 250)
        tr_sub = tr[-n:]                                  # los n días más recientes antes de la validación
        pipe = crear_pipeline(K_FINAL, **best)            # información mutua recalculada con el subconjunto
        pipe.fit(X.iloc[tr_sub], y.iloc[tr_sub])
        filas.append({"n_train": n, "fracción": frac, "fold": fold,
                      "train": perdidas(y.iloc[tr_sub], pipe.predict(X.iloc[tr_sub]))["qlike"].mean(),
                      "validación": perdidas(y.iloc[va], pipe.predict(X.iloc[va]))["qlike"].mean()})

resumen_apr = (pd.DataFrame(filas).groupby("fracción")
               .agg(n_train=("n_train", "mean"),
                    train=("train", "mean"), train_sd=("train", "std"),
                    val=("validación", "mean"), val_sd=("validación", "std")))
display(resumen_apr.round(4))

fig, ax = plt.subplots(figsize=(9, 4.5))
for col, sd, lab in [("train", "train_sd", "train"), ("val", "val_sd", "validación")]:
    ax.plot(resumen_apr["n_train"], resumen_apr[col], "o-", label=lab)
    ax.fill_between(resumen_apr["n_train"], resumen_apr[col] - resumen_apr[sd],
                    resumen_apr[col] + resumen_apr[sd], alpha=0.15)
ax.set_xlabel("Días de entrenamiento (promedio entre folds)"); ax.set_ylabel("QLIKE promedio")
ax.set_title("Curva de aprendizaje del Random Forest (mejores hiperparámetros)")
ax.legend(); plt.tight_layout(); plt.savefig(FIG_DIR / "06_curva_aprendizaje.png", dpi=150); plt.show()

NameError: name 'best' is not defined

## 6. Errores del modelo final por fold

Con los mejores hiperparámetros, en cada uno de los 10 folds:
1. `pipeline.fit(X_train, y_train)`: la información mutua se calcula con el entrenamiento del fold, se eligen sus 16 mejores variables y se entrena el Random Forest.
2. `pipeline.predict(X_train)`: error de **entrenamiento** (dentro de muestra).
3. `pipeline.predict(X_val)`: error de **validación**, en días que el modelo no vio.

Los pronósticos se indexan por el día objetivo τ, igual que los del EGARCH, para compararlos en los mismos días. Una brecha positiva entre validación y entrenamiento es esperable en un Random Forest; una brecha muy grande indicaría sobreajuste.

In [23]:
filas, pron, importancias = [], [], []

for rep, fold, tr, va in CV:
    f = FOLDS[(FOLDS["rep"] == rep) & (FOLDS["fold"] == fold)].iloc[0]

    pipe = crear_pipeline(K_FINAL, **best)               # pipeline completo con información mutua real
    pipe.fit(X.iloc[tr], y.iloc[tr])                     # ajuste solo con el entrenamiento

    h_tr = pipe.predict(X.iloc[tr])                      # error dentro de muestra
    h_va = np.clip(pipe.predict(X.iloc[va]), H_MIN, None)  # pronóstico de validación
    L_tr, L_va = perdidas(y.iloc[tr], h_tr), perdidas(y.iloc[va], h_va)

    filas.append({"rep": rep, "fold": fold, "val_ini": f.val_ini.date(), "val_fin": f.val_fin.date(),
                  "n": len(va),
                  "mse_rf": L_va["mse"].mean(), "qlike_rf": L_va["qlike"].mean(),
                  "mse_train": L_tr["mse"].mean(), "qlike_train": L_tr["qlike"].mean()})
    pron.append(pd.DataFrame({"y": y.iloc[va].values, "h_rf": h_va, "rep": rep, "fold": fold},
                             index=X.index[va]))

    vars_sel = X.columns[pipe.named_steps["seleccion"].get_support()]
    importancias.append(pd.Series(pipe.named_steps["modelo"].feature_importances_,
                                  index=vars_sel, name=f"R{rep}·F{fold}"))
    print(f"R{rep}·F{fold}: QLIKE train = {L_tr['qlike'].mean():.4f} | validación = {L_va['qlike'].mean():.4f}")

res_rf = pd.DataFrame(filas)
res_rf["brecha_qlike"] = res_rf["qlike_rf"] - res_rf["qlike_train"]
pron_rf = pd.concat(pron); pron_rf.index.name = "Date"
imp = pd.concat(importancias, axis=1).fillna(0)

display(res_rf.round(4))
display(res_rf[["qlike_train", "qlike_rf", "mse_train", "mse_rf"]].agg(["mean", "std"]).round(4))

NameError: name 'best' is not defined

## 7. Gráficas de resultados

- QLIKE y MSE de entrenamiento y validación por fold.
- Importancia media de las variables en el Random Forest (una variable no elegida en un fold cuenta como 0).
- Pronóstico del Random Forest frente al Parkinson observado en dos periodos de validación.

In [24]:
etiq = [f"R{a}·F{b}" for a, b in zip(res_rf["rep"], res_rf["fold"])]
x = np.arange(len(etiq))
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for a, met in zip(ax, ["qlike", "mse"]):
    col_val = f"{met}_rf"
    a.bar(x - 0.2, res_rf[f"{met}_train"], 0.4, label="train")
    a.bar(x + 0.2, res_rf[col_val], 0.4, label="validación")
    a.set_xticks(x, etiq, rotation=45); a.set_title(f"{met.upper()} por fold"); a.legend()
plt.tight_layout(); plt.savefig(FIG_DIR / "06_errores_por_fold.png", dpi=150); plt.show()

imp_media = imp.mean(axis=1).sort_values()
imp_media.plot.barh(figsize=(8, 0.3 * len(imp_media) + 1.5),
                    title="Importancia media de las variables en el Random Forest (10 folds)")
plt.tight_layout(); plt.savefig(FIG_DIR / "06_importancia_variables.png", dpi=150); plt.show()

fig, axes = plt.subplots(2, 1, figsize=(13, 7))
for ax, (rep, fold) in zip(axes, [(1, 1), (1, 3)]):
    d = pron_rf.query("rep == @rep and fold == @fold")
    ax.plot(d.index, d["y"], lw=0.6, color="grey", alpha=0.7, label="Parkinson")
    ax.plot(d.index, d["h_rf"], lw=1.3, color="tab:green", label="Pronóstico Random Forest")
    ax.set_title(f"R{rep}·F{fold}: validación {d.index.min().date()} → {d.index.max().date()}")
    ax.set_ylabel("%²"); ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR / "06_pronosticos_folds.png", dpi=150); plt.show()

NameError: name 'res_rf' is not defined

## Guardar resultados


Se guardan con el mismo formato que el modelo econométrico, para usarlos en el notebook de comparación estadística.

In [25]:
res_rf[["rep", "fold", "mse_train", "qlike_train"]].to_csv(RES_DIR / "rf_errores_train_por_fold.csv", index=False)
res_rf.to_csv(RES_DIR / "rf_errores_por_fold.csv", index=False)
pron_rf.to_csv(RES_DIR / "rf_pronosticos_diarios.csv")
imp.to_csv(RES_DIR / "rf_importancia_por_fold.csv")
curva_prof.to_csv(RES_DIR / "rf_curva_validacion_max_depth.csv")
resumen_apr.to_csv(RES_DIR / "rf_curva_aprendizaje.csv")

mejores = {"k": K_FINAL, **best,
           "qlike_validacion_optuna": float(study.best_value),
           "qlike_validacion_final": float(res_rf["qlike_rf"].mean())}
(RES_DIR / "rf_mejores_hiperparametros.json").write_text(json.dumps(mejores, indent=4), encoding="utf-8")

study.trials_dataframe().to_csv(RES_DIR / "rf_optuna_trials.csv", index=False)
joblib.dump(study, RES_DIR / "rf_optuna_study.pkl")
print("Archivos guardados en", RES_DIR)

NameError: name 'res_rf' is not defined

## 9. Conclusiones


- Mejores hiperparámetros y su interpretación (por ejemplo, un `min_samples_leaf` alto indica que el modelo necesita suavizar los días extremos).
- Curva de validación: a partir de qué profundidad el modelo deja de mejorar en validación.
- Curva de aprendizaje: si más datos mejorarían el modelo o si la brecha indica sobreajuste.
- Errores por fold: en qué periodos el error es mayor (2020, 2022) y tamaño de la brecha entre entrenamiento y validación.
- Variables más importantes y su relación con lo observado en la selección de variables.
- Estos pronósticos (`rf_pronosticos_diarios.csv`) se comparan con los del AR(1)-EGARCH en el notebook de comparación estadística.